In [0]:
"""MOSTLY AI preparation, training, reuse, and generated-donor helpers.

Loaded after ``synth_pharos_support`` by the main notebook. The fallback import
also keeps this file directly importable for local tests and editor tooling.
"""


from __future__ import annotations


try:
    from synth_pharos_support import (
        FieldSpec,
        TableSpec,
        _parse_enum_member,
        _prepare_frame,
        det_seed,
        group_from_source,
    )
except ImportError:
    pass  # Names already exist when this file is loaded with Databricks %run.


# --- Mostly ---

from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd


SUBJECT_KEY = "__subject_id"
ROW_KEY = "__row_id"

DEFAULT_TRAIN_TABLES = (
    "person",
    "medical_history",
    "family_cancer_history",
    "comorbidity",
    "medication",
    "imaging",
    "tumour",
    "pathology",
    "pathology_focus",
    "sample",
    "treatment",
    "followup",
    "distant_metastasis",
)

_AUDIT_FIELDS = {"created_at", "updated_at", "date_checked", "clinical_record_id"}
_NUMERIC_TYPES = {"BIGINT", "INT", "SMALLINT", "DOUBLE", "FLOAT", "DECIMAL"}
_DATE_TYPES = {"DATE", "TIMESTAMP"}


def trainable_fields(spec: TableSpec) -> list[FieldSpec]:
    """Fields safe and useful for learned values; keys and narrative text stay out."""
    fields: list[FieldSpec] = []
    for field in spec.fields:
        sql_type = (field.meta.sql_type or "STRING").upper()
        entity_fk = bool(field.meta.fk) and not str(field.meta.fk).startswith("ref_")
        if field.meta.is_pk or entity_fk or field.name in _AUDIT_FIELDS:
            continue
        if field.name == "tumour_group":
            continue  # constant in the breast donor; generated constructively
        if field.enum_cls is not None or sql_type in _NUMERIC_TYPES | _DATE_TYPES | {"BOOLEAN", "BOOL"}:
            fields.append(field)
    return fields


def _coerce_source_feature(
    frame: pd.DataFrame,
    field: FieldSpec,
    min_support: int,
) -> pd.Series:
    series = frame[field.name]
    sql_type = (field.meta.sql_type or "STRING").upper()
    if field.enum_cls is not None:
        def parse(value):
            member = _parse_enum_member(field.enum_cls, value)
            return member.value if member else None

        parsed = series.map(parse)
        support_frame = pd.DataFrame({"value": parsed, SUBJECT_KEY: frame[SUBJECT_KEY]}).dropna()
        support = support_frame.groupby("value")[SUBJECT_KEY].nunique()
        keep = set(support[support >= min_support].index)
        return parsed.where(parsed.isin(keep))
    if sql_type in _NUMERIC_TYPES:
        return pd.to_numeric(series, errors="coerce")
    if sql_type in _DATE_TYPES:
        return pd.to_datetime(series, errors="coerce")
    if sql_type in {"BOOLEAN", "BOOL"}:
        return series.map(lambda value: None if pd.isna(value) else bool(value))
    return pd.Series([None] * len(frame), index=frame.index, dtype="object")


def _select_subjects(root: pd.DataFrame, subject_limit: int | None, seed: int) -> pd.DataFrame:
    root = root.dropna(subset=["pharosid"]).copy()
    root[SUBJECT_KEY] = root["pharosid"].astype(str)
    root = root.drop_duplicates(SUBJECT_KEY, keep="first")
    if subject_limit and len(root) > subject_limit:
        rng = np.random.default_rng(det_seed(seed, "mostly:subjects"))
        chosen = np.sort(rng.choice(len(root), size=subject_limit, replace=False))
        root = root.iloc[chosen]
    return root.reset_index(drop=True)


def _cap_per_subject(frame: pd.DataFrame, cap: int, seed: int, table: str) -> pd.DataFrame:
    if cap <= 0 or frame.empty:
        return frame
    rng = np.random.default_rng(det_seed(seed, f"mostly:cap:{table}"))
    shuffled = frame.iloc[rng.permutation(len(frame))].copy()
    return (
        shuffled.groupby(SUBJECT_KEY, sort=False, group_keys=False)
        .head(cap)
        .sort_index()
        .reset_index(drop=True)
    )


def build_training_frames(
    frames: dict[str, pd.DataFrame],
    registry: dict[str, TableSpec],
    *,
    tables: Iterable[str] = DEFAULT_TRAIN_TABLES,
    subject_limit: int | None = 2_000,
    min_support: int = 10,
    min_rows: int = 50,
    min_subjects: int = 25,
    max_rows_per_subject: int = 25,
    seed: int = 20260818,
) -> dict[str, pd.DataFrame]:
    """Return person-rooted pandas frames accepted by MOSTLY AI local mode."""
    requested = [table for table in tables if table in registry and table in frames]
    if "person" not in requested:
        requested.insert(0, "person")
    person_source = frames.get("person")
    if person_source is None or person_source.empty:
        raise ValueError("MOSTLY AI training requires a non-empty person source frame")

    person = _prepare_frame(person_source, registry["person"])
    if "tumour_group" not in person:
        raise ValueError("person source has no tumour_group for breast conditioning")
    person = person[person["tumour_group"].map(group_from_source) == "breast"]
    person = _select_subjects(person, subject_limit, seed)
    if len(person) < min_subjects:
        raise ValueError(f"only {len(person)} usable breast subjects; need at least {min_subjects}")
    subjects = set(person[SUBJECT_KEY])

    output: dict[str, pd.DataFrame] = {}
    for table in requested:
        source = person.copy() if table == "person" else _prepare_frame(frames[table], registry[table])
        if table != "person":
            if "pharosid" not in source:
                continue
            source = source.dropna(subset=["pharosid"]).copy()
            source[SUBJECT_KEY] = source["pharosid"].astype(str)
            source = source[source[SUBJECT_KEY].isin(subjects)].reset_index(drop=True)
            source = _cap_per_subject(source, max_rows_per_subject, seed, table)
            if len(source) < min_rows or source[SUBJECT_KEY].nunique() < min_subjects:
                continue

        result = pd.DataFrame({SUBJECT_KEY: source[SUBJECT_KEY].astype(str)})
        if table != "person":
            result.insert(0, ROW_KEY, [f"{table}:{index:09d}" for index in range(len(source))])
        for field in trainable_fields(registry[table]):
            if field.name not in source:
                continue
            result[field.name] = _coerce_source_feature(source, field, min_support)
            if result[field.name].notna().sum() == 0:
                result.drop(columns=[field.name], inplace=True)
        if table == "person" or len(result.columns) > 2:
            output[table] = result.reset_index(drop=True)

    if "person" not in output or len(output["person"].columns) <= 1:
        raise ValueError("person training frame has no usable learned fields")
    return output


def _encoding_for(field: FieldSpec) -> str:
    sql_type = (field.meta.sql_type or "STRING").upper()
    if field.enum_cls is not None:
        return "TABULAR_CATEGORICAL"
    if sql_type in _DATE_TYPES:
        return "TABULAR_DATETIME"
    return "AUTO"


def build_generator_tables(
    frames: dict[str, pd.DataFrame],
    registry: dict[str, TableSpec],
    *,
    max_training_time: float = 8.0,
    max_epochs: float = 15.0,
    enable_model_report: bool = False,
) -> list[dict]:
    """Build one person-rooted relational GeneratorConfig ``tables`` list."""
    ordered = ["person"] + [table for table in DEFAULT_TRAIN_TABLES if table != "person" and table in frames]
    ordered += sorted(set(frames) - set(ordered))
    configs: list[dict] = []
    for table in ordered:
        frame = frames[table]
        fields = {field.name: field for field in registry[table].fields}
        columns = []
        for column in frame.columns:
            encoding = "AUTO" if column in {SUBJECT_KEY, ROW_KEY} else _encoding_for(fields[column])
            columns.append({"name": column, "model_encoding_type": encoding})
        config = {
            "name": table,
            "data": frame,
            "primary_key": SUBJECT_KEY if table == "person" else ROW_KEY,
            "columns": columns,
            "tabular_model_configuration": {
                "enable_model_report": enable_model_report,
                "max_training_time": float(max_training_time),
                "max_epochs": float(max_epochs),
            },
        }
        if table != "person":
            config["foreign_keys"] = [{
                "column": SUBJECT_KEY,
                "referenced_table": "person",
                "is_context": True,
            }]
        configs.append(config)
    return configs


def summarize_training_frames(frames: dict[str, pd.DataFrame]) -> dict[str, dict[str, int]]:
    return {
        table: {
            "rows": int(len(frame)),
            "subjects": int(frame[SUBJECT_KEY].nunique()),
            "features": int(len(frame.columns) - (1 if table == "person" else 2)),
        }
        for table, frame in frames.items()
    }


@dataclass
class MostlyDonorPool:
    """Deterministically maps generated breast persons to synthetic MOSTLY subjects."""

    data: dict[str, pd.DataFrame]
    seed: int

    def __post_init__(self):
        root = self.data.get("person")
        self.subjects = [] if root is None else [str(value) for value in root[SUBJECT_KEY].dropna()]
        self._rows: dict[str, dict[str, list[dict]]] = {}
        self._positions: dict[tuple[str, str], int] = {}
        for table, frame in self.data.items():
            if SUBJECT_KEY not in frame:
                continue
            buckets: dict[str, list[dict]] = {}
            for row in frame.to_dict("records"):
                subject = row.get(SUBJECT_KEY)
                if subject is not None and not pd.isna(subject):
                    buckets.setdefault(str(subject), []).append(row)
            self._rows[table] = buckets

    def subject_for(self, pid: str, group: str) -> str | None:
        if group != "breast" or not self.subjects:
            return None
        return self.subjects[det_seed(self.seed, f"mostly:donor:{pid}") % len(self.subjects)]

    def count(self, table: str, pid: str, group: str) -> int | None:
        subject = self.subject_for(pid, group)
        if subject is None or table not in self._rows:
            return None
        return len(self._rows[table].get(subject, []))

    def next_row(self, table: str, pid: str, group: str) -> dict | None:
        subject = self.subject_for(pid, group)
        if subject is None:
            return None
        rows = self._rows.get(table, {}).get(subject, [])
        if not rows:
            return None
        key = (table, pid)
        position = self._positions.get(key, 0)
        self._positions[key] = position + 1
        return rows[position % len(rows)]


def coerce_donor_value(field: FieldSpec, value):
    """Convert MOSTLY/pandas scalars back to the CDM's generated Python values."""
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    sql_type = (field.meta.sql_type or "STRING").upper()
    if field.enum_cls is not None:
        try:
            member = field.enum_cls(value)
        except (ValueError, TypeError):
            return None
        return member.value if member else None
    if sql_type in {"BIGINT", "INT", "SMALLINT"}:
        try:
            return int(round(float(value)))
        except (TypeError, ValueError):
            return None
    if sql_type in {"DOUBLE", "FLOAT", "DECIMAL"}:
        try:
            return float(value)
        except (TypeError, ValueError):
            return None
    if sql_type == "DATE":
        parsed = pd.to_datetime(value, errors="coerce")
        return None if pd.isna(parsed) else parsed.date().isoformat()
    if sql_type == "TIMESTAMP":
        parsed = pd.to_datetime(value, errors="coerce")
        return None if pd.isna(parsed) else parsed.to_pydatetime().isoformat()
    if sql_type in {"BOOLEAN", "BOOL"}:
        return bool(value)
    return str(value)


# --- Mostly Runtime ---

import gc
import hashlib
import json
import os
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path



@dataclass(frozen=True)
class MostlyRunConfig:
    mode: str = "off"
    seed: int = 20260818
    breast_persons: int = 600
    training_subjects: int = 2_000
    tables: tuple[str, ...] = DEFAULT_TRAIN_TABLES
    max_training_time: float = 8.0
    max_epochs: float = 15.0
    allow_cpu: bool = False
    model_path: Path | None = None
    work_dir: Path | None = None
    report_dir: Path | None = None
    generator_name: str | None = None


def run_mostly(
    frames,
    registry,
    config: MostlyRunConfig,
) -> tuple[dict | None, dict]:
    """Train/reuse a breast relational generator and return generated donor frames."""
    if config.mode == "off":
        return None, {"mode": "off", "trained": False, "generated": False}
    if config.mode not in {"train", "reuse"}:
        raise ValueError(f"unsupported MOSTLY AI mode: {config.mode}")
    if config.model_path is None:
        raise ValueError("MOSTLY AI train/reuse requires model_path")

    import torch
    from mostlyai.sdk import MostlyAI

    gpu_ok = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if gpu_ok else None
    if not gpu_ok and not config.allow_cpu:
        raise RuntimeError(
            "MOSTLY AI requires CUDA for this run; pass allow_cpu=True only when intentional"
        )

    model_path = Path(config.model_path)
    model_path.parent.mkdir(parents=True, exist_ok=True)
    work_dir = Path(config.work_dir or model_path.parent / "mostly_work")
    work_dir.mkdir(parents=True, exist_ok=True)
    report_dir = Path(config.report_dir or model_path.parent.parent / "reports")
    report_dir.mkdir(parents=True, exist_ok=True)
    mostly = MostlyAI(local=True, local_dir=str(work_dir))
    training_summary = None
    started = time.time()

    if config.mode == "reuse":
        if not model_path.is_file():
            raise RuntimeError(f"MOSTLY AI model is missing: {model_path}")
        generator = mostly.generators.import_from_file(str(model_path))
        training_status = "REUSED"
    else:
        if not frames:
            raise RuntimeError("MOSTLY AI training requires source frames")
        training_frames = build_training_frames(
            frames,
            registry,
            tables=config.tables,
            subject_limit=config.training_subjects,
            min_support=10,
            min_rows=50,
            min_subjects=25,
            max_rows_per_subject=25,
            seed=config.seed,
        )
        training_summary = summarize_training_frames(training_frames)
        print("MOSTLY AI training frames:", json.dumps(training_summary, indent=2))
        table_configs = build_generator_tables(
            training_frames,
            registry,
            max_training_time=config.max_training_time,
            max_epochs=config.max_epochs,
            enable_model_report=False,
        )
        generator_name = config.generator_name or (
            "pharos_cdm2_breast_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        )
        trained = mostly.train(
            config={"name": generator_name, "tables": table_configs}, start=False
        )
        trained.training.start()
        trained.training.wait(progress_bar=True)
        generator = mostly.generators.get(trained.id)
        raw_status = getattr(generator, "training_status", None)
        training_status = raw_status.value if hasattr(raw_status, "value") else str(raw_status)
        try:
            trained.training.logs(file_path=str(report_dir / "mostly_train_logs.zip"))
        except Exception as exc:
            print("MOSTLY AI logs export skipped:", exc)
        if str(training_status).upper() != "DONE":
            raise RuntimeError(f"MOSTLY AI training did not finish DONE: {training_status}")
        pending = model_path.with_name(f"{model_path.stem}_{config.seed}_{time.time_ns()}.zip")
        try:
            trained.export_to_file(str(pending))
            os.replace(pending, model_path)
        finally:
            pending.unlink(missing_ok=True)
        del training_frames, table_configs
        gc.collect()

    donor_persons = max(config.breast_persons * 2, 100)
    synthetic = mostly.generate(generator, size={"person": donor_persons})
    data = synthetic.data()
    if "person" not in data or len(data["person"]) == 0:
        raise RuntimeError("MOSTLY AI returned no person donor rows")
    generated_counts = {table: int(len(frame)) for table, frame in data.items()}
    report = {
        "mode": config.mode,
        "trained": config.mode == "train",
        "generated": True,
        "training_status": training_status,
        "training_seconds": round(time.time() - started, 1),
        "gpu_available": gpu_ok,
        "gpu_name": gpu_name,
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "mostlyai_version": __import__("mostlyai.sdk", fromlist=["__version__"]).__version__,
        "model_path": str(model_path),
        "model_sha256": hashlib.sha256(model_path.read_bytes()).hexdigest(),
        "training_frames": training_summary,
        "generated_counts": generated_counts,
        "donor_persons_requested": donor_persons,
        "max_training_time_minutes_per_table": config.max_training_time,
        "max_epochs_per_table": config.max_epochs,
    }
    return data, report